# 05 — Video Mode 3: Agentic Multi-Scene Video (Colab-native)

The notebook port of `agent/agent.py` (spec 05) — no ComfyUI, no vendor API. Given a character +
a short prompt, it:

1. **EXPAND** — a *local* LLM (Ollama in this same Colab, `qwen2.5:3b` by default) expands the
   prompt into a cinematic scene plan (JSON). Skip the LLM entirely by pasting your own plan.
2. **KEYFRAMES** — one FLUX.1-dev + character-LoRA still per scene boundary (N scenes → N+1 keyframes).
3. **VIDEOS** — LTX-Video 0.9.8-13B interpolates each consecutive keyframe pair into a clip.
4. **STITCH** — ffmpeg concat → `final.mp4`.

**Resumable:** the state machine persists to `state.json` on Drive at every step. If the GPU
disconnects mid-run, start a fresh Colab, rerun cells 1–5 (cheap), then run the stage cells
(6 → 7 → 8) in order — each stage checks `state.json` and skips completed work.

**VRAM hygiene:** Ollama runs only during EXPAND and is killed before GPU models load. The FLUX
pipeline is deleted before the LTX pipeline loads. Stages never share GPU residency.

**GPU:** A100 recommended (FLUX keyframes ~24 GB, LTX clips ~28 GB, in separate stages). 40 GB
works with the offload strategies built in. Below that, expect pain — this is a 2-stage GPU job.

**Runtime budget:** ~5 keyframes + 4 clips on an A100 ≈ 40–80 min total, plus first-run model
downloads (both stages cache to Drive, so only the first run pays it).

## 1. Config + mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Config — change these ────────────────────────────────────────────────
CHARACTER_NAME = 'Yuna'
TRIGGER_TOKEN  = 'sks_vyuna'
LORA_STRENGTH  = 1.4     # validated in 02a A/B sweep 2026-09-06
OLLAMA_MODEL   = 'qwen2.5:3b'    # local LLM for scene planning (7b on A100, see §9)
N_SCENES_MIN, N_SCENES_MAX = 3, 5
CLIP_FRAMES    = 81              # 4k+1; 81 ≈ 3.4 s @ 24 fps per scene segment
CLIP_W, CLIP_H = 832, 480
KF_W, KF_H     = 1024, 1024      # keyframe still resolution (LTX resizes internally)
# ─────────────────────────────────────────────────────────────────────────

import os, time, uuid
DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
LORA_PATH  = f'{DRIVE_BASE}/loras/{CHARACTER_NAME}_flux.safetensors'
AG_ROOT    = f'{DRIVE_BASE}/outputs/videos/{CHARACTER_NAME}/agentic'
RUN_ID     = time.strftime('%Y%m%d_%H%M%S')   # or set a fixed id to RESUME an existing run
RUN_DIR    = f'{AG_ROOT}/{RUN_ID}'
os.makedirs(f'{RUN_DIR}/keyframes', exist_ok=True)
os.makedirs(f'{RUN_DIR}/clips', exist_ok=True)
os.environ['HF_HOME'] = f'{DRIVE_BASE}/models/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

print(f'Character : {CHARACTER_NAME} | trigger {TRIGGER_TOKEN}')
print(f'LoRA      : {LORA_PATH} (present: {os.path.exists(LORA_PATH)})')
print(f'Run dir   : {RUN_DIR}')

## 2. Install (uv, system env)

In [ ]:
!pip install -q uv
!uv pip install --system -q --reinstall-package diffusers \
    "diffusers>=0.35.0" transformers accelerate safetensors huggingface_hub ftfy
!apt-get install -y -q ffmpeg >/dev/null 2>&1 || true
!ffmpeg -version 2>/dev/null | head -1

import torch, diffusers
print('torch', torch.__version__, '| diffusers', diffusers.__version__)

# torchao: Colab ships 0.10.0 but the peft uv pulls requires >=0.16.0 and its
# is_torchao_available() check raises instead of returning False. We don't use
# torchao (quantization lib) for bf16 FLUX/Wan/LTX + LoRA, so remove it to keep
# peft's LoRA injection path clean.
!uv pip uninstall --system -y torchao 2>/dev/null || true
# ── CUDA-torch guard ────────────────────────────────────────────────────────
# uv can silently re-resolve deps and swap Colab's CUDA torch for the PyPI CPU
# wheel ("Torch not compiled with CUDA enabled" at load time). Detect + repair
# here, BEFORE we commit to a multi-GB model download.
import torch as _t
print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
if not _t.cuda.is_available():
    import subprocess
    ver = _t.__version__.split('+')[0]
    print(f'CPU-only torch {ver} detected (uv swapped it). Reinstalling the CUDA build from cu124...')
    subprocess.run(f'uv pip install --system "torch=={ver}" torchvision '
                   '--index-url https://download.pytorch.org/whl/cu124',
                   shell=True, check=True)
    raise RuntimeError(
        'CUDA torch reinstalled on disk. Now: Runtime > Restart runtime, then re-run '
        'this install cell + the model-load cell. (The running kernel still holds the '
        'old CPU torch in memory, so the restart is required — do not skip it.)')


## 3. Local LLM — Ollama in this Colab (EXPAND stage only)
No vendor API: Ollama runs as a background process in this same VM. `qwen2.5:3b` (~1.9 GB) is the
default — small enough to coexist with everything and good enough for structured JSON planning.
First run downloads the model to `~/.ollama` (ephemeral — re-pull on new sessions, ~1 min at A100
network). The server is **killed before any GPU diffusion model loads** (cell 6).

In [ ]:
import subprocess, os, time

def ensure_ollama():
    if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
        print('Installing Ollama...')
        subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True,
                       capture_output=True)
    # run the server detached (survives the cell; we kill it in cell 6)
    log = open('/content/ollama.log', 'w')
    subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT)
    time.sleep(3)
    # pull the model
    print(f'Pulling {OLLAMA_MODEL} (first run only)...')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    # quick smoke test
    out = subprocess.run(['ollama', 'run', OLLAMA_MODEL, 'Reply with the single word: ready'],
                         capture_output=True, text=True, timeout=120).stdout.strip()
    print('Ollama OK. Model said:', out[-60:])

ensure_ollama()

## 4. LLM provider + scene planner (JSON schema from spec 05)
The Ollama provider is a plain HTTP client to the local server. The **Claude API fallback is
commented out** at the bottom — uncomment if you ever want a stronger planner (needs
`ANTHROPIC_API_KEY`, and it sends your prompts/character notes to Anthropic — local stays default
on purpose).

The planner returns the exact spec-05 schema: `{title, tone, scenes:[{id, description, setting,
action, start_keyframe_prompt, end_keyframe_prompt, transition_to_next}]}`. Keyframes are derived
as N+1 deduplicated boundaries (scene i's end = scene i+1's start).

In [ ]:
import json, urllib.request, re

class OllamaLocal:
    """Minimal client for the Ollama server on localhost (runs in this same Colab VM)."""
    def __init__(self, model=OLLAMA_MODEL, base_url='http://localhost:11434'):
        self.model, self.base = model, base_url

    def complete(self, system, user, json_mode=False):
        body = {
            'model': self.model,
            'messages': [
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user},
            ],
            'stream': False,
            'format': 'json' if json_mode else None,
        }
        body = {k: v for k, v in body.items() if v is not None}
        req = urllib.request.Request(
            f'{self.base}/api/chat',
            data=json.dumps(body).encode(),
            headers={'Content-Type': 'application/json'},
        )
        with urllib.request.urlopen(req, timeout=600) as r:
            return json.loads(r.read())['message']['content']

llm = OllamaLocal()

# ── spec-05 planner prompts ────────────────────────────────────────────────
EXPAND_SYSTEM = (
    "You are a cinematic script supervisor for AI video generation. "
    "Expand the user's brief prompt into a structured scene plan. "
    "Return valid JSON only, no prose."
)
EXPAND_USER = """
User prompt: "{prompt}"
Character: {character_name} — trigger token: {trigger}

Return a JSON object with this exact schema:
{{
  "title": "short title",
  "tone": "cinematic tone description",
  "scenes": [
    {{
      "id": 1,
      "description": "what happens in this scene (2-3 sentences, cinematic)",
      "setting": "location/environment",
      "action": "what the character does",
      "start_keyframe_prompt": "detailed IMAGE prompt for the opening frame",
      "end_keyframe_prompt": "detailed IMAGE prompt for the closing frame",
      "transition_to_next": "how this scene flows into the next"
    }}
  ]
}}

Rules:
- {n_min}–{n_max} scenes.
- Each keyframe prompt is an IMAGE prompt (no motion words) and MUST start with the trigger token "{trigger}"
- Keyframe prompts describe composition, lighting, expression, setting concretely. Do NOT describe the face.
- scene[i].end_keyframe_prompt and scene[i+1].start_keyframe_prompt must depict the SAME frame (continuity).
- Keep scenes sequentially logical.
"""

def plan_scenes(user_prompt, character_name, trigger):
    raw = llm.complete(
        EXPAND_SYSTEM,
        EXPAND_USER.format(prompt=user_prompt, character_name=character_name,
                           trigger=trigger, n_min=N_SCENES_MIN, n_max=N_SCENES_MAX),
        json_mode=True,
    )
    # be tolerant of stray prose around the JSON
    m = re.search(r'\{.*\}', raw, re.S)
    plan = json.loads(m.group(0) if m else raw)
    plan['scenes'] = plan.get('scenes', [])
    for i, sc in enumerate(plan['scenes']):
        sc['id'] = i + 1
        for k in ('start_keyframe_prompt', 'end_keyframe_prompt'):
            if not sc.get(k, '').startswith(trigger):
                sc[k] = f"{trigger}, {sc.get(k, '')}"
    return plan

def keyframes_from_plan(plan):
    """N scenes -> N+1 deduplicated boundary keyframes (spec 05 step 2)."""
    scenes = plan['scenes']
    kfs = [{'scene_id': scenes[0]['id'], 'position': 'start', 'prompt': scenes[0]['start_keyframe_prompt']}]
    for i, sc in enumerate(scenes):
        is_last = (i == len(scenes) - 1)
        kfs.append({'scene_id': sc['id'], 'position': 'end',
                    'prompt': sc['end_keyframe_prompt'] if is_last else scenes[i+1]['start_keyframe_prompt']})
    return kfs

print('planner + keyframe derivation ready.')

# ─────────────────────────────────────────────────────────────────────────
# CLAUDE API FALLBACK (commented — opt-in, non-local):
# ─────────────────────────────────────────────────────────────────────────
# import os
# ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')  # or Colab Secrets
# def claude_complete(system, user, json_mode=False):
#     req = urllib.request.Request(
#         'https://api.anthropic.com/v1/messages',
#         data=json.dumps({
#             'model': 'claude-sonnet-4-6',
#             'max_tokens': 4096,
#             'system': system,
#             'messages': [{'role': 'user', 'content': user}],
#         }).encode(),
#         headers={'Content-Type': 'application/json',
#                  'x-api-key': ANTHROPIC_API_KEY,
#                  'anthropic-version': '2023-06-01'},
#     )
#     with urllib.request.urlopen(req, timeout=600) as r:
#         return json.loads(r.read())['content'][0]['text']
# llm_complete = claude_complete   # then use in plan_scenes instead of llm.complete

# NOTE: plan_scenes calls llm.complete — for the Claude fallback, either subclass
# OllamaLocal with that method, or sed-replace the one call site above.

## 5. EXPAND — generate (or paste) the scene plan, then initialize `state.json`
Two ways in:
- **A (default):** set `USER_PROMPT` and let the local LLM plan it.
- **B (skip LLM / quality escape hatch):** set `MANUAL_PLAN_JSON` to a plan conforming to the
  schema in §4 (hand-written or from another tool). The LLM is never called.

Resuming an existing run? Set `RUN_ID` in cell 1 to the existing timestamp — this cell will detect
the existing `state.json` and reuse the stored plan instead of re-expanding.

In [ ]:
import json, os

USER_PROMPT = 'Yuna sneaks through a moonlit forest to retrieve a stolen artifact, and escapes into a rain-soaked city.'
MANUAL_PLAN_JSON = None   # or a dict/str conforming to the spec-05 schema (skips the LLM)

state_file = f'{RUN_DIR}/state.json'

if os.path.exists(state_file):
    state = json.load(open(state_file))
    print(f"Resuming existing run (state: {state['state']}). Keeping stored plan.")
    plan = state['plan']
else:
    if MANUAL_PLAN_JSON:
        plan = json.loads(MANUAL_PLAN_JSON) if isinstance(MANUAL_PLAN_JSON, str) else MANUAL_PLAN_JSON
        print('Using manually supplied plan (LLM skipped).')
    else:
        print('Expanding prompt with local Ollama LLM (this is the only LLM call in the run)...')
        plan = plan_scenes(USER_PROMPT, CHARACTER_NAME, TRIGGER_TOKEN)
    state = {
        'state': 'keyframes',   # EXPAND is complete once we have a plan
        'run_id': RUN_ID,
        'character': CHARACTER_NAME,
        'prompt': USER_PROMPT,
        'plan': plan,
        'keyframe_paths': [],
        'clip_paths': [],
        'final_path': None,
    }

plan['keyframes'] = keyframes_from_plan(plan)
state['plan'] = plan
json.dump(state, open(state_file, 'w'), indent=2)

# free the LLM from VRAM now — diffusion stages come next
import subprocess
subprocess.run(['pkill', '-f', 'ollama serve'], capture_output=True)
import torch
torch.cuda.empty_cache()

print(f"\nPlan: '{plan['title']}' — {len(plan['scenes'])} scenes, {len(plan['keyframes'])} keyframes")
for sc in plan['scenes']:
    print(f"  [{sc['id']}] {sc['setting']} — {sc['action'][:60]}")
print(f"\nstate saved → {state_file}")

## 6. Stage KEYFRAMES — FLUX.1-dev + LoRA (skip-safe, frees VRAM at the end)
One still per keyframe boundary (N+1 total), seeds 400+i for variety. Saves to
`<run>/keyframes/kf_<i>.png`. When it finishes it **deletes the FLUX pipeline and frees VRAM** —
so the LTX stage in cell 7 can load without OOM. Re-running this cell after a crash only generates
the missing keyframes.

In [ ]:
import json, os, torch, time, gc
from pathlib import Path

state = json.load(open(state_file))
if state['state'] not in ('keyframes',):
    print(f'KEYFRAMES stage not due (state={state["state"]}). Skipping — rerun only if you want to redo keyframes.')
else:
    # Version-robust strength setter (diffusers >=0.32 uses set_adapters(names, weights))
    def set_strength(pipe, s):
        # adapter is 'default_0' (what load_lora_weights registers);
        # diffusers >=0.32: set_adapters(names, weights); fallback to one-arg form
        try:
            pipe.set_adapters(['default_0'], [s])
        except Exception:
            pipe.set_adapters([s])

    # ── load FLUX.1-dev + LoRA (same strategy as 02a, compact) ──────────────
    from diffusers import FluxPipeline
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU ~{vram_gb:.0f} GB')
    hf_token = ''
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        pass
    pipe = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-dev', dtype=torch.bfloat16,
                                        token=hf_token or None)
    if vram_gb >= 60:
        pipe.to('cuda')
    else:
        from diffusers.hooks import apply_group_offloading
        onload, offload = torch.device('cuda'), torch.device('cpu')
        for comp in (pipe.transformer, pipe.text_encoder, pipe.text_encoder_2, pipe.vae):
            apply_group_offloading(comp, offload_device=offload, onload_device=onload,
                                   offload_type='leaf_level', use_stream=True)
    pipe.vae.enable_slicing(); pipe.vae.enable_tiling()
    pipe.load_lora_weights(LORA_PATH)
    set_strength(pipe, LORA_STRENGTH)
    print('FLUX + LoRA loaded.')

    kf_list = state['plan']['keyframes']
    kf_paths = list(state.get('keyframe_paths', []))
    for i, kf in enumerate(kf_list):
        dest = f'{RUN_DIR}/keyframes/kf_{i}.png'
        if i < len(kf_paths) and kf_paths[i] and os.path.exists(kf_paths[i]):
            print(f'  kf {i+1}/{len(kf_list)} done, skipping.')
            continue
        print(f'  generating keyframe {i+1}/{len(kf_list)} ({kf["scene_id"]}/{kf["position"]}): {kf["prompt"][:60]}...')
        g = torch.Generator(device='cuda' if vram_gb >= 60 else 'cpu').manual_seed(400 + i)
        im = pipe(prompt=kf['prompt'], width=KF_W, height=KF_H, num_inference_steps=28,
                  guidance_scale=3.5, generator=g, max_sequence_length=512).images[0]
        im.save(dest)
        kf_paths.append(dest)
        state['keyframe_paths'] = kf_paths
        json.dump(state, open(state_file, 'w'), indent=2)   # persist per keyframe

    state['state'] = 'videos'
    json.dump(state, open(state_file, 'w'), indent=2)

    # ── free FLUX before the LTX stage ──────────────────────────────────────
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    print(f'✅ Keyframes complete: {len(kf_paths)}. FLUX freed. State → videos.')

# quick look at what we have so far
from IPython.display import display, HTML
import base64
def _thumb(p, s=240):
    from PIL import Image
    im = Image.open(p).convert('RGB'); im.thumbnail((s, s))
    from io import BytesIO
    b = BytesIO(); im.save(b, 'JPEG', quality=80)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}" title="{os.path.basename(p)}">'
kfs = sorted(Path(f'{RUN_DIR}/keyframes').glob('kf_*.png'), key=lambda p: int(p.stem.split('_')[1]))
if kfs:
    display(HTML('<h4>Keyframes</h4><div style="display:flex;flex-wrap:wrap;gap:8px">' +
                 ''.join(_thumb(str(p)) for p in kfs) + '</div>'))


## 7. Stage VIDEOS — LTX-Video interpolation (skip-safe, frees VRAM at the end)
Loads LTX-Video 0.9.8-13B-distilled, interpolates each consecutive keyframe pair (kf_i → kf_i+1)
using the scene's `transition_to_next` as the motion prompt. Saves `<run>/clips/seg_<i>.mp4`,
persists per clip, then frees the pipeline. Same resume semantics as cell 6.

In [ ]:
import json, os, torch, time, gc
from pathlib import Path
from PIL import Image

state = json.load(open(state_file))
if state['state'] not in ('videos',):
    print(f'VIDEOS stage not due (state={state["state"]}). Skipping.')
else:
    from diffusers import LTXConditionPipeline
    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
    from diffusers.utils import export_to_video

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU ~{vram_gb:.0f} GB')
    pipeline = LTXConditionPipeline.from_pretrained('Lightricks/LTX-Video-0.9.8-13B-distilled',
                                                    dtype=torch.bfloat16)
    pipeline.vae.enable_tiling()
    if vram_gb >= 60:
        pipeline.to('cuda')
    else:
        from diffusers.hooks import apply_group_offloading
        onload, offload = torch.device('cuda'), torch.device('cpu')
        pipeline.transformer.enable_group_offload(onload_device=onload, offload_device=offload,
                                                  offload_type='leaf_level', use_stream=True)
        apply_group_offloading(pipeline.text_encoder, onload_device=onload, offload_device=offload,
                               offload_type='block_level', num_blocks_per_group=2)
        apply_group_offloading(pipeline.vae, onload_device=onload, offload_device=offload,
                               offload_type='leaf_level')
    print('LTX loaded.')

    kf_paths = state['keyframe_paths']
    clip_paths = list(state.get('clip_paths', []))
    scenes = state['plan']['scenes']
    neg = 'worst quality, inconsistent motion, blurry, jittery, distorted'
    def r32(x): return int(x) // 32 * 32
    h, w = r32(CLIP_H), r32(CLIP_W)

    for i in range(len(kf_paths) - 1):
        dest = f'{RUN_DIR}/clips/seg_{i}.mp4'
        if i < len(clip_paths) and clip_paths[i] and os.path.exists(clip_paths[i]):
            print(f'  seg {i+1} done, skipping.')
            continue
        scene_idx = min(i, len(scenes) - 1)
        motion = f'{TRIGGER_TOKEN}, {scenes[scene_idx].get("transition_to_next", "")}, cinematic motion'
        print(f'  generating clip {i+1}/{len(kf_paths)-1} (scene {scene_idx+1}): {motion[:70]}...')
        cond = [
            LTXVideoCondition(image=Image.open(kf_paths[i]).convert('RGB'), frame_index=0),
            LTXVideoCondition(image=Image.open(kf_paths[i+1]).convert('RGB'),
                              frame_index=CLIP_FRAMES - 1),
        ]
        g = torch.Generator().manual_seed(1000 + i)
        frames = pipeline(
            conditions=cond, prompt=motion, negative_prompt=neg,
            width=w, height=h, num_frames=CLIP_FRAMES,
            timesteps=[1000, 993, 987, 981, 975, 909, 725, 0.03],
            decode_timestep=0.05, decode_noise_scale=0.025, image_cond_noise_scale=0.0,
            guidance_scale=1.0, guidance_rescale=0.7,
            generator=g, output_type='pil',
        ).frames[0]
        export_to_video(frames, dest, fps=24)
        clip_paths.append(dest)
        state['clip_paths'] = clip_paths
        json.dump(state, open(state_file, 'w'), indent=2)   # persist per clip
        print(f'    → {dest}')

    state['state'] = 'stitch'
    json.dump(state, open(state_file, 'w'), indent=2)

    del pipeline
    gc.collect()
    torch.cuda.empty_cache()
    print(f'✅ Clips complete: {len(clip_paths)}. LTX freed. State → stitch.')

for c in sorted(Path(f'{RUN_DIR}/clips').glob('seg_*.mp4')):
    print(' ', c)

## 8. Stage STITCH — ffmpeg concat → `final.mp4`
Plain concat (stream copy) is the reliable default. The xfade-crossfade variant is in §9 if the
hard cuts bother you — it needs a full re-encode.

In [ ]:
import json, os, subprocess
from pathlib import Path

state = json.load(open(state_file))
if state['state'] not in ('stitch',):
    print(f'STITCH stage not due (state={state["state"]}). Skipping.')
else:
    clip_paths = state['clip_paths']
    final_path = f'{RUN_DIR}/final.mp4'
    concat_list = f'{RUN_DIR}/concat.txt'
    with open(concat_list, 'w') as f:
        for p in clip_paths:
            f.write(f"file '{Path(p).resolve()}'\n")
    r = subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_list,
                        '-c', 'copy', final_path], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'ffmpeg stitch failed: {r.stderr[-800:]}')
    state['final_path'] = final_path
    state['state'] = 'done'
    json.dump(state, open(state_file, 'w'), indent=2)
    print(f'✅ final video → {final_path}')

from IPython.display import Video, display
if state.get('final_path') and os.path.exists(state['final_path']):
    display(Video(state['final_path'], width=720))

## 9. Log to metadata + commented alternates

In [ ]:
import json, os, time
meta_path = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}/metadata.json'
os.makedirs(os.path.dirname(meta_path), exist_ok=True)
meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {'name': CHARACTER_NAME}
state = json.load(open(state_file))
meta.setdefault('video_log', []).append({
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'mode': 'mode3_agentic',
    'run_id': RUN_ID,
    'title': state['plan'].get('title'),
    'scenes': len(state['plan']['scenes']),
    'final': state.get('final_path'),
    'state': state['state'],
})
json.dump(meta, open(meta_path, 'w'), indent=2)
print(f'metadata.json updated — run {RUN_ID} ({state["state"]}) logged.')

# ─────────────────────────────────────────────────────────────────────────
# Alternates (commented):

# A) Bigger local planner: OLLAMA_MODEL = 'qwen2.5:7b' in cell 1 (needs ~5 GB;
#    fine on A100, tight on free-tier RAM). Or the Claude API fallback in §4.

# B) Hero-scene quality pass: regenerate ONE scene's clip with Wan FLF2V (03b)
#    instead of LTX — best two-frame quality — then re-stitch. Swap that one
#    entry in state['clip_paths'] and rerun cell 8.

# C) xfade crossfade stitch (re-encodes; ~0.5 s overlap between segments):
#    import subprocess
#    # build filter_complex with cumulative offsets:
#    # [0][1]xfade=transition=fade:duration=0.5:offset=D1-0.5[v01];
#    # [v01][2]xfade=transition=fade:duration=0.5:offset=D1+D2-1.0[v012]; ...
#    # Get each clip's duration D_i with: ffprobe -v error -show_entries
#    #   format=duration -of csv=p=0 <clip>
#    # Full example in agent/agent.py notes / ffmpeg docs. Plain concat (default)
#    # is safer; xfade offsets are fiddly with variable-length segments.

# D) Longer clips: CLIP_FRAMES = 121 (5 s) or 161 (~6.7 s) in cell 1 — VRAM and
#    time scale ~linearly. Or chain 03a-style: last frame of seg N as kf for N+1.

# E) Video LoRA: if identity drifts through motion, train a Wan/LTX character
#    video LoRA (spec 01 step 4) and load it in the VIDEOS stage via
#    pipeline.load_lora_weights(...) before the interpolation loop.

print('\n✅ 05 complete (or resumed to completion). state.json is the source of truth for the run.')